# fxbt — Example Backtest

Demonstrates the full workflow:
1. Load data (CSV or BQuant)
2. Generate signals
3. Run backtest with cost model + position sizing
4. View results and tear sheet

In [ ]:
import sys, os
sys.path.append(os.path.realpath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from fxbt2 import Backtest, CSVLoader, BQuantLoader
from fxbt2 import signals, metrics, report
from fxbt2.backtest.positions import vol_target, equal_weight
from fxbt2.costs import FixedSpreadModel, SpreadCostModel

## 1. Load Data

### Option A: CSV (local development)
Expects one CSV per pair in `data/` directory, or a combined CSV.
Bloomberg terminal exports work out of the box.

### Option B: BQuant (Bloomberg environment)
Uncomment the BQuantLoader block when running on BQuant.

In [ ]:
PAIRS = ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'USDCAD']
START = '2018-01-01'
END   = '2024-12-31'
FREQ  = '1D'

# --- Option A: CSV ---
# loader = CSVLoader(data_dir='../data/')
# df = loader.load(PAIRS, start=START, end=END, freq=FREQ)

# --- Option B: BQuant ---
# loader = BQuantLoader(load_forwards=True, load_vol=True)
# df = loader.load(PAIRS, start=START, end=END, freq=FREQ)

# --- Demo: synthetic random data (replace with real data above) ---
np.random.seed(42)
dates = pd.date_range(START, END, freq='B', tz='UTC')
spot_starts = {'EURUSD': 1.10, 'GBPUSD': 1.25, 'USDJPY': 110.0, 'AUDUSD': 0.70, 'USDCAD': 1.32}

frames = []
for pair, start_px in spot_starts.items():
    rets = np.random.normal(0, 0.006, len(dates))
    prices = start_px * np.exp(np.cumsum(rets))
    frames.append(pd.DataFrame({
        'pair': pair,
        'close': prices,
        'open': prices * (1 + np.random.normal(0, 0.001, len(dates))),
        'high': prices * (1 + np.abs(np.random.normal(0, 0.003, len(dates)))),
        'low':  prices * (1 - np.abs(np.random.normal(0, 0.003, len(dates)))),
        'fwd_points': np.random.normal(0, 0.0005, len(dates)),
    }, index=dates))

df = pd.concat(frames).sort_index()
print(f'Loaded {df["pair"].nunique()} pairs, {len(df)} rows')
df.head()

In [ ]:
# Pivot to wide format for signal generation
from fxbt2.data.base import DataLoader
prices = DataLoader.to_wide(df, field='close')
fwd_pts = DataLoader.to_wide(df, field='fwd_points') if 'fwd_points' in df.columns else None

print('Prices shape:', prices.shape)
prices.tail(3)

## 2. Generate Signals

In [ ]:
# --- Momentum signal (20-day lookback) ---
mom_signals = signals.momentum(prices, lookback=20)

# --- Carry signal (requires fwd_points) ---
# carry_signals = signals.carry(fwd_pts, prices)

# --- MA crossover ---
# cross_signals = signals.crossover(prices, fast=10, slow=50)

# --- Combine: momentum filtered by vol regime ---
vol_filter = signals.vol_regime(prices, lookback=20, high_vol_percentile=75)
filtered_signals = mom_signals * vol_filter

print('Signals preview:')
filtered_signals.tail(3)

## 3. Run Backtest

In [ ]:
# Fixed spread cost model (no bid/ask data needed)
cost_model = FixedSpreadModel(slippage_bps=0.5)

# Vol-targeting position sizing (10% ann. vol target)
sizer = lambda s, p: vol_target(s, p, target_vol=0.10, freq=FREQ)

bt = Backtest(
    data=prices,
    signals=filtered_signals,
    cost_model=cost_model,
    sizer=sizer,
    freq=FREQ,
    name='20D Momentum + Vol Filter',
)

result = bt.run()
print('Backtest complete.')

## 4. Results

In [ ]:
result.summary()

In [ ]:
result.tearsheet()

## 5. Compare Multiple Strategies

In [ ]:
# Run a second strategy to compare against
cross_signals = signals.crossover(prices, fast=10, slow=50)

bt2 = Backtest(
    data=prices,
    signals=cross_signals,
    cost_model=cost_model,
    sizer=sizer,
    freq=FREQ,
    name='10/50 MA Crossover',
)
result2 = bt2.run()

# Side-by-side comparison
result.compare_to(result2)

In [ ]:
# Or compare directly via metrics.compare()
metrics.stats.compare(
    (result.returns, '20D Momentum'),
    (result2.returns, 'MA Crossover'),
    freq=FREQ,
)

## 6. Walk-Forward Analysis

In [ ]:
def signal_fn(train_prices, test_prices):
    """Re-generate momentum signal using only training data context."""
    # Concatenate train+test so rolling windows have context,
    # then slice to test period only
    combined = pd.concat([train_prices, test_prices])
    full_signals = signals.momentum(combined, lookback=20)
    return full_signals.loc[test_prices.index]

wf_result = bt.walk_forward(
    train_periods=252,   # 1 year training
    test_periods=63,     # 1 quarter test
    signal_fn=signal_fn,
)

wf_result.summary()